In [1]:
!pip -q install scikit-learn

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [3]:
os.makedirs("data/08_model_inputs_updated", exist_ok=True)

In [5]:
dataset = pd.read_csv("dataset_v2_aligned_finbert_updated.csv")
embeddings = np.load("finbert_embeddings_raw_updated.npy")

print("Dataset shape:", dataset.shape)
print("Embeddings shape:", embeddings.shape)

Dataset shape: (2247, 26)
Embeddings shape: (2247, 768)


In [6]:
assert len(dataset) == embeddings.shape[0], "Row mismatch between dataset and embeddings."

print("Rows match correctly.")
print(dataset[["ticker", "filing_date", "filing_type"]].head())
print("Embedding dimension:", embeddings.shape[1])

Rows match correctly.
  ticker filing_date filing_type
0   AAPL  2019-01-30        10-Q
1   AAPL  2019-05-01        10-Q
2   AAPL  2019-07-31        10-Q
3   AAPL  2020-01-29        10-Q
4   AAPL  2020-05-01        10-Q
Embedding dimension: 768


In [7]:
dataset["filing_date"] = pd.to_datetime(dataset["filing_date"], errors="coerce")
dataset = dataset.dropna(subset=["filing_date"]).copy()

sort_idx = dataset.sort_values(["filing_date", "ticker"]).index
dataset = dataset.loc[sort_idx].reset_index(drop=True)
embeddings = embeddings[sort_idx.to_numpy()]

print("Dataset shape after sorting:", dataset.shape)
print("Embeddings shape after sorting:", embeddings.shape)

Dataset shape after sorting: (2247, 26)
Embeddings shape after sorting: (2247, 768)


In [8]:
split_date = pd.Timestamp("2023-01-01")

train_mask = dataset["filing_date"] < split_date
test_mask = dataset["filing_date"] >= split_date

train_df = dataset.loc[train_mask].copy().reset_index(drop=True)
test_df = dataset.loc[test_mask].copy().reset_index(drop=True)

X_train_emb = embeddings[train_mask.to_numpy()]
X_test_emb = embeddings[test_mask.to_numpy()]

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train embedding shape:", X_train_emb.shape)
print("Test embedding shape:", X_test_emb.shape)

Train rows: 1460
Test rows: 787
Train embedding shape: (1460, 768)
Test embedding shape: (787, 768)


In [9]:
print("Train filing date range:", train_df["filing_date"].min(), "to", train_df["filing_date"].max())
print("Test filing date range:", test_df["filing_date"].min(), "to", test_df["filing_date"].max())

Train filing date range: 2019-01-25 00:00:00 to 2022-12-29 00:00:00
Test filing date range: 2023-01-05 00:00:00 to 2024-12-13 00:00:00


In [10]:
finbert_scaler = StandardScaler()
X_train_emb_scaled = finbert_scaler.fit_transform(X_train_emb)
X_test_emb_scaled = finbert_scaler.transform(X_test_emb)

print("Scaled train shape:", X_train_emb_scaled.shape)
print("Scaled test shape:", X_test_emb_scaled.shape)

Scaled train shape: (1460, 768)
Scaled test shape: (787, 768)


In [11]:
N_COMPONENTS = 15

finbert_pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_train_pca = finbert_pca.fit_transform(X_train_emb_scaled)
X_test_pca = finbert_pca.transform(X_test_emb_scaled)

print("Train PCA shape:", X_train_pca.shape)
print("Test PCA shape:", X_test_pca.shape)
print("Total explained variance:", finbert_pca.explained_variance_ratio_.sum())

Train PCA shape: (1460, 15)
Test PCA shape: (787, 15)
Total explained variance: 0.80960137


In [12]:
pca_cols = [f"finbert_pca_{i+1}" for i in range(N_COMPONENTS)]

train_pca_df = pd.DataFrame(X_train_pca, columns=pca_cols)
test_pca_df = pd.DataFrame(X_test_pca, columns=pca_cols)

train_final = pd.concat([train_df.reset_index(drop=True), train_pca_df], axis=1)
test_final = pd.concat([test_df.reset_index(drop=True), test_pca_df], axis=1)

print("Train final shape:", train_final.shape)
print("Test final shape:", test_final.shape)
train_final.head()

Train final shape: (1460, 41)
Test final shape: (787, 41)


,ticker,cik,filing_date,filing_type,accession_number,year,quarter,cik_nolead,acc_nodash,mda_path,...,finbert_pca_6,finbert_pca_7,finbert_pca_8,finbert_pca_9,finbert_pca_10,finbert_pca_11,finbert_pca_12,finbert_pca_13,finbert_pca_14,finbert_pca_15
0,ADBE,796343,2019-01-25,10-K,0000796343-19-000019,2019,1,796343,79634319000019,ADBE_20190125_10-K_000079634319000019.txt,...,-0.422460,1.637497,-2.221855,-2.597467,-1.179479,1.316340,0.186844,-1.068573,-0.193942,-1.177217
1,KLAC,319201,2019-01-29,10-Q,0000319201-19-000009,2019,1,319201,31920119000009,KLAC_20190129_10-Q_000031920119000009.txt,...,1.007965,-6.826082,2.097694,-1.308432,-3.716791,-2.309129,-1.547493,1.501166,-2.425273,-5.103899
2,SBUX,829224,2019-01-29,10-Q,0000829224-19-000011,2019,1,829224,82922419000011,SBUX_20190129_10-Q_000082922419000011.txt,...,4.765640,9.146583,-1.179768,-4.651479,-0.690860,2.284552,-0.138787,-1.506142,-0.163595,1.107608
3,AAPL,320193,2019-01-30,10-Q,0000320193-19-000010,2019,1,320193,32019319000010,AAPL_20190130_10-Q_000032019319000010.txt,...,3.274701,1.384812,5.469298,-1.881741,3.564189,-2.304030,6.557678,-8.790899,4.022316,-4.799107
4,CMCSA,1166691,2019-01-31,10-K,0001166691-19-000005,2019,1,1166691,116669119000005,CMCSA_2019-01-31_10-K_0001166691-19-000005_STR...,...,5.231605,2.317092,-4.249330,-1.114698,-3.196644,2.044073,-0.085526,-2.530400,-5.048415,3.563553


In [13]:
print([c for c in train_final.columns if c.startswith("finbert_pca_")])
print("Number of PCA columns in train:", len([c for c in train_final.columns if c.startswith("finbert_pca_")]))
print("Number of PCA columns in test:", len([c for c in test_final.columns if c.startswith("finbert_pca_")]))

['finbert_pca_1', 'finbert_pca_2', 'finbert_pca_3', 'finbert_pca_4', 'finbert_pca_5', 'finbert_pca_6', 'finbert_pca_7', 'finbert_pca_8', 'finbert_pca_9', 'finbert_pca_10', 'finbert_pca_11', 'finbert_pca_12', 'finbert_pca_13', 'finbert_pca_14', 'finbert_pca_15']
Number of PCA columns in train: 15
Number of PCA columns in test: 15


In [14]:
train_final.to_csv("data/08_model_inputs_updated/train_dataset_with_finbert_pca.csv", index=False)
test_final.to_csv("data/08_model_inputs_updated/test_dataset_with_finbert_pca.csv", index=False)

np.save("data/08_model_inputs_updated/train_finbert_pca.npy", X_train_pca)
np.save("data/08_model_inputs_updated/test_finbert_pca.npy", X_test_pca)

with open("data/08_model_inputs_updated/finbert_scaler_train.pkl", "wb") as f:
    pickle.dump(finbert_scaler, f)

with open("data/08_model_inputs_updated/finbert_pca_train.pkl", "wb") as f:
    pickle.dump(finbert_pca, f)

print("Saved all split-aware FinBERT PCA outputs.")

Saved all split-aware FinBERT PCA outputs.


In [15]:
print("Train final rows:", len(train_final))
print("Test final rows:", len(test_final))
print("Train missing PCA values:", train_final[pca_cols].isna().sum().sum())
print("Test missing PCA values:", test_final[pca_cols].isna().sum().sum())
print("Explained variance:", finbert_pca.explained_variance_ratio_.sum())

Train final rows: 1460
Test final rows: 787
Train missing PCA values: 0
Test missing PCA values: 0
Explained variance: 0.80960137


In [16]:
price_features = [
    "past_return_10d",
    "abs_past_return_10d",
    "past_realized_vol_5d",
    "past_realized_vol_10d"
]

lm_features = [
    "lm_negative",
    "lm_positive",
    "lm_uncertainty",
    "lm_net_sentiment"
]

finbert_features = pca_cols.copy()

target_col = "future_realized_vol_10d"

print("Price features:", price_features)
print("LM features:", lm_features)
print("FinBERT features:", finbert_features[:5], "...")
print("Target:", target_col)

Price features: ['past_return_10d', 'abs_past_return_10d', 'past_realized_vol_5d', 'past_realized_vol_10d']
LM features: ['lm_negative', 'lm_positive', 'lm_uncertainty', 'lm_net_sentiment']
FinBERT features: ['finbert_pca_1', 'finbert_pca_2', 'finbert_pca_3', 'finbert_pca_4', 'finbert_pca_5'] ...
Target: future_realized_vol_10d


In [17]:
from google.colab import files

files.download("data/08_model_inputs_updated/train_dataset_with_finbert_pca.csv")
files.download("data/08_model_inputs_updated/test_dataset_with_finbert_pca.csv")
files.download("data/08_model_inputs_updated/finbert_scaler_train.pkl")
files.download("data/08_model_inputs_updated/finbert_pca_train.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>